# Apache Spark — Core Concepts & Optimization

Training notebook covering Spark internals: distributed processing, lazy evaluation, execution model, performance monitoring (Serverless tools and the classic-compute Spark UI & logs), transformations, streaming, and optimization techniques.

## 1. Distributed Processing Architecture

### How Spark Distributes Work

```
┌─────────────────────────────────────────────────────────┐
│                     DRIVER PROGRAM                      |
│  • Schedules tasks                • Collects results    │
└──────────────────────┬──────────────────────────────────┘
                       │  distributes tasks
       ┌───────────────┼───────────────┐
       ▼               ▼               ▼
 ┌───────────┐   ┌───────────┐   ┌───────────┐
 │ EXECUTOR 1│   │ EXECUTOR 2│   │ EXECUTOR 3│
 │ ┌───┬───┐ │   │ ┌───┬───┐ │   │ ┌───┬───┐ │
 │ │T1 │T2 │ │   │ │T3 │T4 │ │   │ │T5 │T6 │ │
 │ └───┴───┘ │   │ └───┴───┘ │   │ └───┴───┘ │
 └───────────┘   └───────────┘   └───────────┘
    Worker 1        Worker 2        Worker 3
```

### Key Concepts

| Component | Role |
|---|---|
| **Driver** | Orchestrates the application, creates DAG, assigns tasks |
| **Executor** | JVM process on worker node, runs tasks & stores data |
| **Task** | Smallest unit of work — operates on **one partition** |
| **Partition** | Chunk of data processed in parallel |
| **Cluster Manager** | Allocates resources (managed by Databricks on Serverless) |

> **Key insight:** More partitions = more parallelism (up to available cores)

In [0]:
from pyspark.sql.functions import col, sum as _sum

# --- Lazy: Nothing executes yet ---
orders = spark.read.table("samples.tpch.orders")



In [0]:

# Chain of transformations — all lazy
result = (
    orders
    .filter(col("o_orderstatus") == "F")
    .groupBy("o_orderpriority")
    .agg(_sum("o_totalprice").alias("total_revenue"))
    .orderBy(col("total_revenue").desc())
)

print("✅ No computation happened yet — only a plan was built")
print("─" * 55)

# View the execution plan
result.explain(mode="formatted")

In [0]:
# --- Trigger execution with an ACTION ---
# This creates: 1 Job → multiple Stages → Tasks
# Check the Spark UI after running!

display(result)  # ACTION: triggers the full DAG

In [0]:
#Why it does laze evaluation , Because of the lazy evaluation, Spark can optimize the query plan before executing it. For example, it can push down filters to the data source, or it can optimize joins to reduce the amount of data that needs to be transferred between nodes.
df=spark.read.table("samples.tpch.orders")


In [0]:
df.filter(col("o_orderstatus") == "F")


In [0]:
df.display()


### Transformations vs Actions

| Transformations (Lazy) | Actions (Trigger Execution) |
|---|---|
| `select`, `filter`, `withColumn` | `count`, `show`, `collect` |
| `groupBy`, `join`, `orderBy` | `write`, `save`, `display` |
| `map`, `flatMap`, `union` | `first`, `take`, `foreach` |
| Build a **logical plan** — nothing runs yet | **Trigger** the DAG → jobs → stages → tasks |

### Execution Hierarchy

```
Action (e.g. df.count())
 └── Job  (one per action)
      ├── Stage 1  (separated by shuffles)
      │    ├── Task 1  (one per partition)
      │    └── Task 2
      └── Stage 2
           ├── Task 3
           └── Task 4
```

### Execution Plans

| Plan | What It Shows |
|---|---|
| **Parsed Logical Plan** | Raw user intent |
| **Analyzed Logical Plan** | Resolved columns & types |
| **Optimized Logical Plan** | Catalyst optimizer rewrites |
| **Physical Plan** | Actual execution strategy (scans, joins, exchanges) |

> **Key insight:** Spark optimizes the **entire chain** of transformations before running anything

### Execution Plan Phases — What Happens at Each Plan

The Catalyst optimizer turns your DataFrame code into 4 plans. `explain(mode="extended")` prints all of them.

```
Your code → ① Parsed → ② Analyzed → ③ Optimized → ④ Physical → Jobs / Stages / Tasks
            (syntax)  (catalog)   (rules)       (how to run)
```

| Plan | Activity | What to look for in the example below |
|---|---|---|
| **① Parsed (Unresolved) Logical** | Builds a tree from your code exactly as written. Tables & columns are **not checked yet**. | `'UnresolvedRelation [samples, tpch, orders]`, columns prefixed with `'` (`'o_totalprice`), two separate `'Filter` nodes, `100 * 10` not computed |
| **② Analyzed Logical** | Resolves names against **Unity Catalog**: checks tables/columns exist, assigns data types and unique IDs, adds casts. Errors like `UNRESOLVED_COLUMN` come from here. | `Relation samples.tpch.orders[...] parquet`, `o_totalprice#2104`, output types `revenue: decimal(28,2)`, `cast(... as decimal(18,2))` |
| **③ Optimized Logical** | Applies rule-based rewrites — same result, less work. | **Constant folding**: `100 * 10` → `1000.00` • **Combine filters**: two filters → one `AND` • **Predicate pushdown**: filter moved *below* the join • **Column pruning**: `Project [o_custkey, o_totalprice]` before the join • **Null filtering**: `isnotnull(o_custkey)` added for the join key |
| **④ Physical** | Chooses **how** to execute: join algorithm, shuffles, scan details, Photon operators. AQE can still change it at runtime (`isFinalPlan=false`). | `PhotonScan ... ReadSchema` (only 3 columns read), `DataFilters` / `DictionaryFilters` (filter pushed into the file scan), `ShuffleExchange hashpartitioning(o_custkey, 200)` (= shuffle = new stage), `ShuffledHashJoin` vs `BroadcastHashJoin`, `partial_sum` → shuffle → `finalmerge_sum` |

> Each **Exchange** in the physical plan is a stage boundary you will see in the Spark UI after an action runs.

## 3. Narrow vs Wide Transformations

### Comparison

| | Narrow | Wide |
|---|---|---|
| **Data movement** | None — each partition works independently | **Shuffle** — data moves across nodes |
| **Stage boundary** | Same stage | Creates a **new stage** |
| **Examples** | `select`, `filter`, `map`, `withColumn`, `union` | `groupBy`, `join`, `orderBy`, `distinct`, `repartition` |
| **Performance** | ✅ Fast, pipelined | ⚠️ Expensive — network + disk I/O |

### Visual

```
NARROW (no shuffle)              WIDE (shuffle = stage boundary)

 Partition 1 → Partition 1       Partition 1 ─┬─→ Partition A
 Partition 2 → Partition 2       Partition 2 ─┼─→ Partition B
 Partition 3 → Partition 3       Partition 3 ─┴─→ Partition C
                                  ▲ data redistributed
```

> **Key insight:** Every wide transformation = a shuffle = network I/O = **the #1 performance bottleneck in Spark**

In [0]:
from pyspark.sql.functions import col, upper

orders = spark.read.table("samples.tpch.orders")

# --- NARROW transformations (no shuffle, same stage) ---
narrow_result = (
    orders
    .select("o_orderkey", "o_orderstatus", "o_totalprice")
    .filter(col("o_totalprice") > 1000)
    .withColumn("status_upper", upper(col("o_orderstatus")))
)

print("=== NARROW Plan (no Exchange/shuffle) ===")
narrow_result.explain()

print("\n" + "═" * 55)

# --- WIDE transformations (shuffle required) ---
wide_result = (
    orders
    .groupBy("o_orderstatus")
    .count()
    .orderBy(col("count").desc())
)

print("\n=== WIDE Plan (Exchange = shuffle!) ===")
wide_result.explain()

## 4. Performance Monitoring on Serverless

> ⚠️ **Serverless does NOT have the traditional Spark UI.** Use these tools instead. On **classic compute**, the Spark UI and driver/executor logs are available — see **Section 4b**.

### How to Monitor on Serverless

| Tool | What It Shows | How to Access |
|---|---|---|
| **`explain()`** | Physical plan — scans, joins, shuffles, filters | `df.explain()` or `df.explain(mode="formatted")` |
| **`explain("cost")`** | Plan + estimated row counts & sizes | `df.explain(mode="cost")` |
| **Query Profile** | Visual execution plan in SQL Editor | Run query in SQL Editor → click **"Query Profile"** |
| **Query History** | Past queries, duration, rows scanned | Left sidebar → **Query History** |

### Reading an Execution Plan

```
AdaptiveSparkPlan          ← AQE wrapper (always present)
 +- PhotonSort              ← orderBy()
    +- Exchange              ← SHUFFLE (wide transformation!)
       +- PhotonGroupingAgg  ← groupBy().agg()
          +- PhotonScan      ← table/file scan (filters pushed here)
```

### What to Look For in `explain()`

| Look For | Meaning |
|---|---|
| **PhotonScan** with `DataFilters` | ✅ Predicate pushdown is working |
| **Exchange** | ⚠️ Shuffle happening — wide transformation |
| **PhotonBroadcastHashJoin** | ✅ Small table broadcast (no shuffle) |
| **SortMergeJoin** | ⚠️ Both sides shuffled — expensive |
| **AdaptiveSparkPlan** | ✅ AQE is active |
| **ReadSchema** with fewer columns | ✅ Column pruning is working |

> 💡 **Tip:** For visual execution plans, run your query in the **SQL Editor** and click **"Query Profile"**

In [0]:
from pyspark.sql.functions import col, count, avg

# --- On Serverless: use explain() to inspect execution plans ---
lineitem = spark.read.table("samples.tpch.lineitem")

heavy_query = (
    lineitem
    .filter(col("l_shipdate") >= "1995-01-01")
    .groupBy("l_returnflag", "l_linestatus")
    .agg(
        count("*").alias("count_orders"),
        avg("l_quantity").alias("avg_qty"),
        avg("l_discount").alias("avg_discount")
    )
    .orderBy("l_returnflag", "l_linestatus")
)

# 1️⃣ Formatted plan — shows each operator with details
print("=== FORMATTED PLAN ===")
heavy_query.explain(mode="formatted")

print("\n" + "═" * 55)

# 2️⃣ Cost plan — includes estimated row counts
print("\n=== COST PLAN (with row estimates) ===")
heavy_query.explain(mode="cost")

# 3️⃣ Execute and display results
print("\n=== RESULTS ===")
display(heavy_query)

print("\n💡 On Serverless: explain() is your primary debugging tool")
print("   For visual plans: run the SQL in the SQL Editor → Query Profile")

## 4b. Spark UI & Logs on Classic Compute

> Applies to **classic all-purpose / job clusters** (Dedicated or No-isolation access mode). Serverless and Standard (shared) compute do not expose the Spark UI — use Section 4 instead.

### How to Open the Spark UI

| From | Path |
|---|---|
| **Notebook cell** | Expand **"Spark Jobs"** under a cell's output → click **View** next to a job |
| **Cluster page** | **Compute** → select cluster → **Spark UI** tab |
| **Job run** | **Jobs & Pipelines** → run → task → **Spark UI** link |
| **Terminated cluster** | Spark UI stays available (event-log based) for a limited time after termination |

### Spark UI Tabs

| Tab | What It Shows | Use It To |
|---|---|---|
| **Jobs** | One row per action; duration, stages, status | Find the slow action |
| **Stages** | Tasks per stage, shuffle read/write, spill, task time distribution (min/median/max) | Detect **skew** (max >> median) and **spill** |
| **Storage** | Cached DataFrames/tables, memory vs disk usage | Verify caching worked / free memory |
| **Environment** | Spark configs, JVM & classpath | Confirm `spark.sql.*` settings |
| **Executors** | Cores, memory, GC time, failed tasks, **stdout/stderr per executor** | Spot GC pressure, lost executors, executor logs |
| **SQL / DataFrame** | Visual physical plan with **runtime metrics** per operator (rows, bytes, time) | See actual rows per Exchange/Join, AQE final plan |
| **Structured Streaming** | Input rate, process rate, batch duration per stream | Check whether a stream keeps up |

### Execution Hierarchy in the UI

```
Jobs tab      →  Job 12  (display(result))
Stages tab    →    ├── Stage 30  Scan + partial agg  (200 tasks)   shuffle write
              →    └── Stage 31  Final agg + sort    (AQE: 4 tasks) shuffle read
SQL tab       →  Query 7: visual DAG with rows/bytes per operator
```

### Red Flags to Look For

| Symptom (where) | Likely Cause | Fix |
|---|---|---|
| Max task time >> median (Stages → Summary Metrics) | **Data skew** | AQE skew join, salting, filter hot keys |
| **Spill (Memory/Disk)** > 0 (Stages) | Partitions too large for executor memory | More shuffle partitions, larger nodes |
| Large **Shuffle Read/Write** (Stages) | Wide transformations, big joins | Broadcast small tables, filter/prune earlier |
| High **GC Time** (Executors) | Memory pressure, large objects | Bigger memory, fewer cached objects |
| Thousands of tiny tasks (Stages) | Too many small files/partitions | `OPTIMIZE`, coalesce, AQE coalescing |
| Failed/killed executors (Executors) | OOM, spot instance loss | Check executor stderr / driver log4j |

### Driver & Executor Logs

| Log | Where | Contains |
|---|---|---|
| **Driver `stdout`** | Compute → cluster → **Driver logs** | `print()` output from driver-side code |
| **Driver `stderr`** | Compute → cluster → **Driver logs** | Python tracebacks, warnings |
| **Driver `log4j`** | Compute → cluster → **Driver logs** | Spark/JVM internals: scheduling, AQE decisions, OOM errors |
| **Executor `stdout`/`stderr`** | Spark UI → **Executors** tab → per-executor links | Output/errors from code running on workers (e.g. UDFs) |
| **Event log** | Compute → cluster → **Event log** | Cluster lifecycle: start, resize, autoscaling, terminations |
| **Cluster log delivery** | Cluster config → **Advanced → Logging** (UC Volume or DBFS path) | Persists driver/executor/init-script logs after termination |

> 💡 **Tip:** Label your jobs with `spark.conf.set("spark.job.description", "...")` (next cell) so they are easy to find in the Jobs and SQL tabs.

In [0]:
from pyspark.sql.functions import col, sum as _sum, count

# spark.job.description sets the 'Description' column in the Spark UI Jobs / SQL tabs.
# Works on Dedicated AND Standard (shared) access mode — no sparkContext needed.

orders = spark.read.table("samples.tpch.orders")
lineitem = spark.read.table("samples.tpch.lineitem")

# Job 1: narrow + wide (filter -> groupBy -> orderBy)
spark.conf.set("spark.job.description", "dox_01_revenue_by_priority")
revenue = (
    orders.filter(col("o_orderstatus") == "F")
    .groupBy("o_orderpriority")
    .agg(_sum("o_totalprice").alias("total_revenue"))
    .orderBy(col("total_revenue").desc())
)
display(revenue)

# Job 2: join -> look for SortMergeJoin / ShuffledHashJoin / BroadcastHashJoin in the SQL tab
spark.conf.set("spark.job.description", "dox_02_lineitems_per_order_status")
joined = (
    lineitem.join(orders, lineitem.l_orderkey == orders.o_orderkey)
    .groupBy("o_orderstatus")
    .agg(count("*").alias("line_count"))
)
display(joined)

# Reset so later cells are not mislabelled
spark.conf.unset("spark.job.description")

print("""
💡 Open the Spark UI (Compute -> this cluster -> Spark UI):
   • Jobs tab   : Description column shows 'dox_01_...' / 'dox_02_...'
   • Stages tab : compare shuffle read/write and task time min/median/max
   • SQL tab    : open the query -> rows per operator, Exchange nodes, join strategy
""")

In [0]:
from pyspark.sql.functions import col, when, count

# --- Create an intentionally skewed key and aggregate on it ---
# ~90% of rows land on key 0 -> one shuffle partition gets most of the data.
skewed = (
    spark.range(5_000_000)
    .withColumn("key", when(col("id") % 10 < 9, 0).otherwise(col("id") % 1000))
)

spark.conf.set("spark.job.description", "dox_03_skewed_groupBy")

skew_result = (
    skewed.groupBy("key")
    .agg(count("*").alias("rows"))
    .orderBy(col("rows").desc())
)
display(skew_result.limit(10))
spark.conf.unset("spark.job.description")

print("""💡 In the Spark UI (classic):
   Stages tab -> open the shuffle-read stage -> 'Summary Metrics for Tasks'
   • Duration / Shuffle Read Size: Max is far above the Median => SKEW
   • Check 'Spill (Memory)' / 'Spill (Disk)' columns for spilling tasks
   SQL tab -> AQE may show 'AQEShuffleRead' with coalesced/skew-split partitions
""")

## 5. Batch vs Streaming Workloads

### Comparison

| | Batch | Streaming |
|---|---|---|
| **Data** | Bounded, finite dataset | Continuous, unbounded data |
| **Processing** | Start → process all → end | Always running, processes as data arrives |
| **Latency** | Minutes to hours | Seconds to minutes |
| **API** | `spark.read` → `df.write` | `spark.readStream` → `df.writeStream` |
| **Use Cases** | ETL, reports, backfills | Real-time dashboards, alerts, CDC |

### Structured Streaming — Unified API

```
 Batch:     spark.read.table("src")  →  transform  →  df.write.table("tgt")
 Stream:    spark.readStream("src")  →  transform  →  df.writeStream("tgt")
                                        ▲
                                  same DataFrame API!
```

### Trigger Modes

| Trigger | Behavior |
|---|---|
| `availableNow=True` | Process all available data, then stop (like batch but incremental) |
| `processingTime='10 seconds'` | Micro-batch every 10 seconds |
| `once=True` | Process one micro-batch, then stop |
| *(default)* | Next micro-batch as soon as previous finishes |

In [0]:
# ========== SHARED CONFIG for Section 5 (Batch + Streaming) ==========
# Source: 20 CSV part files (TPC-H customer, ~750K rows, with header)
SOURCE_PATH = "abfss://inboundfiles@unityms.dfs.core.windows.net/customer"

CATALOG, SCHEMA = "one_dbx_training", "default"

# Batch target
BATCH_TABLE = f"{CATALOG}.{SCHEMA}.customer_batch"

# Streaming targets — one table per trigger mode
STREAM_TABLE_AVAILABLE_NOW   = f"{CATALOG}.{SCHEMA}.customer_stream_available_now"
STREAM_TABLE_PROCESSING_TIME = f"{CATALOG}.{SCHEMA}.customer_stream_processing_time"
STREAM_TABLE_DEFAULT         = f"{CATALOG}.{SCHEMA}.customer_stream_default"

# Streaming state (checkpoints + Auto Loader schema) lives in a UC Volume
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.checkpoints")
CHECKPOINT_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/checkpoints/customer"

# Explicit schema — required for the plain file streaming source
CUSTOMER_SCHEMA = """
    c_custkey INT, c_name STRING, c_address STRING, c_nationkey INT,
    c_phone STRING, c_acctbal DOUBLE, c_mktsegment STRING, c_comment STRING
"""

print(f"Source     : {SOURCE_PATH}")
print(f"Batch table: {BATCH_TABLE}")
print(f"Checkpoints: {CHECKPOINT_ROOT}")

### 5a. Batch Processing

* `spark.read` reads **every file** in the folder each time it runs — it keeps **no memory** of what was already loaded.
* `mode("overwrite")` → full reload every run (safe but re-reads everything).
* `mode("append")` → re-running loads **the same files again** → duplicate rows.
* To make batch incremental you must build the tracking yourself (watermark columns, file lists, `MERGE`) — or use streaming (5b).

In [0]:
from pyspark.sql.functions import col, current_timestamp

# ========== BATCH READ ==========
# Bounded read: lists ALL files in SOURCE_PATH at execution time
batch_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .schema(CUSTOMER_SCHEMA)
    .load(SOURCE_PATH)
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_ingested_at", current_timestamp())
)

# ========== BATCH WRITE ==========
(batch_df.write
    .format("delta")
    .mode("overwrite")          # full load
    .saveAsTable(BATCH_TABLE))

print(f"✅ Batch load #1 complete → {BATCH_TABLE}")
print(f"   Rows in table: {spark.table(BATCH_TABLE).count():,}")
display(spark.table(BATCH_TABLE).limit(5))

In [0]:
# ========== BATCH RE-RUN ==========
# Run the SAME batch read again. No new files arrived, but Spark has no state,
# so it re-reads all 20 files. With append, every row is loaded a second time.
rows_before = spark.table(BATCH_TABLE).count()

reprocess_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .schema(CUSTOMER_SCHEMA)
    .load(SOURCE_PATH)
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_ingested_at", current_timestamp())
)

(reprocess_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(BATCH_TABLE))

rows_after = spark.table(BATCH_TABLE).count()
print(f"Rows before re-run : {rows_before:,}")
print(f"Rows read again    : {rows_after - rows_before:,}   ← entire dataset re-loaded")
print(f"Rows after re-run  : {rows_after:,}")

# Each load batch is visible by _ingested_at; each customer now appears twice
display(
    spark.table(BATCH_TABLE)
    .groupBy("_ingested_at").count()
    .orderBy("_ingested_at")
)

# Delta history shows one WRITE per batch run
display(spark.sql(f"DESCRIBE HISTORY {BATCH_TABLE}").select("version", "timestamp", "operation", "operationParameters", "operationMetrics"))

### 5b. Streaming Processing (Incremental)

Same source folder, but `spark.readStream` + a **checkpoint** remembers which files were already processed. Re-running only picks up **new** files.

**Read options covered**

| Source | Key options |
|---|---|
| File source `format("csv")` | explicit `schema` (required), `header`, `maxFilesPerTrigger` |
| Auto Loader `format("cloudFiles")` | `cloudFiles.format`, `cloudFiles.schemaLocation`, `cloudFiles.inferColumnTypes` or explicit `schema`, `cloudFiles.maxFilesPerTrigger` |

**Write / trigger options covered** — each writes to its **own table + checkpoint**

| Trigger | Target table | Behavior |
|---|---|---|
| `availableNow=True` | `customer_stream_available_now` | Process everything available (in multiple batches), then stop |
| `processingTime="10 seconds"` | `customer_stream_processing_time` | Micro-batch every 10s, runs until stopped |
| *(default — no trigger)* | `customer_stream_default` | Next micro-batch as soon as the previous one finishes |

> `trigger(once=True)` is deprecated — use `availableNow=True`. `continuous` triggers do not support file sources/Delta sinks.

In [0]:
# Read: Auto Loader (cloudFiles) — infers the schema and tracks processed files
df_available_now = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaLocation", f"{CHECKPOINT_ROOT}/available_now/_schema")
    .option("cloudFiles.maxFilesPerTrigger", 5)
    .load(SOURCE_PATH)
)

# Write: availableNow → process all files that exist now, then stop
query = (
    df_available_now.writeStream
    .option("checkpointLocation", f"{CHECKPOINT_ROOT}/available_now")
    .trigger(availableNow=True)
    .toTable(STREAM_TABLE_AVAILABLE_NOW)
)
query.awaitTermination()

print("Rows:", spark.table(STREAM_TABLE_AVAILABLE_NOW).count())

In [0]:
# Read: plain CSV file source — schema must be given
df_processing_time = (
    spark.readStream
    .format("csv")
    .option("header", "true")
    .option("maxFilesPerTrigger", 5)
    .schema(CUSTOMER_SCHEMA)
    .load(SOURCE_PATH)
)

# Write: processingTime → a micro-batch every 10 seconds, runs until stopped
query = (
    df_processing_time.writeStream
    .option("checkpointLocation", f"{CHECKPOINT_ROOT}/processing_time")
    .trigger(processingTime="10 seconds")
    .toTable(STREAM_TABLE_PROCESSING_TIME)
)
query.processAllAvailable()   # wait until current files are loaded
query.stop()                  # otherwise it keeps running

print("Rows:", spark.table(STREAM_TABLE_PROCESSING_TIME).count())

In [0]:
# Read: Auto Loader with an explicit schema instead of inference
df_default = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", f"{CHECKPOINT_ROOT}/default/_schema")
    .schema(CUSTOMER_SCHEMA)
    .load(SOURCE_PATH)
)

# Write: no trigger → next micro-batch starts as soon as the previous one finishes
query = (
    df_default.writeStream
    .option("checkpointLocation", f"{CHECKPOINT_ROOT}/default")
    .toTable(STREAM_TABLE_DEFAULT)
)
query.processAllAvailable()
query.stop()

print("Rows:", spark.table(STREAM_TABLE_DEFAULT).count())

In [0]:
# Re-run Stream 1 with the SAME checkpoint → already-processed files are skipped
rows_before = spark.table(STREAM_TABLE_AVAILABLE_NOW).count()

df_rerun = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaLocation", f"{CHECKPOINT_ROOT}/available_now/_schema")
    .load(SOURCE_PATH)
)

query = (
    df_rerun.writeStream
    .option("checkpointLocation", f"{CHECKPOINT_ROOT}/available_now")
    .trigger(availableNow=True)
    .toTable(STREAM_TABLE_AVAILABLE_NOW)
)
query.awaitTermination()

rows_after = spark.table(STREAM_TABLE_AVAILABLE_NOW).count()
print("Rows before:", rows_before)
print("Rows after :", rows_after)
print("New rows   :", rows_after - rows_before, "(0 = nothing reloaded)")

In [0]:
# --- Managing active streaming queries ---
# List all active streams
for q in spark.streams.active:
    print(f"Active stream: name={q.name}, id={q.id}, status={q.status}")

if not spark.streams.active:
    print("No active streams (expected — we only showed the pattern above)")

print("""\n💡 Common stream management commands:
   spark.streams.active           → list running streams
   query.stop()                   → stop a specific stream
   query.awaitTermination()       → block until stream ends
   query.lastProgress             → latest micro-batch metrics
""")

## 6. Spark Optimization

### Optimization Techniques at a Glance

| Technique | What It Does | When to Use |
|---|---|---|
| **Predicate Pushdown** | Pushes filters to data source — reads less data | Always — automatic with Spark |
| **Column Pruning** | Reads only selected columns | Use `select()` instead of `select("*")` |
| **AQE** | Dynamically adjusts at runtime | Enabled by default in Databricks |
| **Partition Tuning** | Right-sizes parallelism | Skew, too many/few tasks |

### Adaptive Query Execution (AQE)

AQE optimizes **at runtime** based on actual data statistics:

| AQE Feature | What It Does |
|---|---|
| **Coalesce Shuffle Partitions** | Merges small partitions → fewer tasks |
| **Skew Join Optimization** | Splits skewed partitions into sub-partitions |

> 💡 AQE is **enabled by default** in Databricks — look for `AdaptiveSparkPlan` in explain()

In [0]:
from pyspark.sql.functions import col

orders = spark.read.table("samples.tpch.orders")

# --- BAD: Reads all columns, filters late ---
bad = orders.select("*").filter(col("o_orderstatus") == "F")

# --- GOOD: Select only needed columns, filter early ---
good = orders.select("o_orderkey", "o_totalprice", "o_orderstatus").filter(col("o_orderstatus") == "F")

print("=== BAD: select(*) then filter ===")
bad.explain()

print("\n=== GOOD: select + filter (pushdown) ===")
good.explain()

print("\n💡 Notice: Spark pushes the filter INTO the scan in both cases")
print("   But selecting fewer columns = less data read from disk")

In [0]:
from pyspark.sql.functions import col, sum as _sum

# --- AQE: Adaptive Query Execution ---
# AQE is always ON in Databricks (serverless & classic)
print("✅ AQE is enabled by default in Databricks")
print("   • Coalesces small shuffle partitions")
print("   • Converts sort-merge to broadcast joins at runtime")
print("   • Handles skewed join partitions")

# Run a query with shuffle — AQE will optimize partition count
orders = spark.read.table("samples.tpch.orders")

agg_result = (
    orders
    .groupBy("o_orderstatus", "o_orderpriority")
    .agg(_sum("o_totalprice").alias("total"))
    .orderBy(col("total").desc())
)

# Check the plan: look for "AdaptiveSparkPlan"
print("\n=== Plan with AQE ===")
agg_result.explain(mode="formatted")

print("\n💡 Look for 'AdaptiveSparkPlan' at the top of the plan")
print("   After execution it shows 'isFinalPlan=true' with optimized stages")

display(agg_result)